# LAB 04 - Experiment 5: n-gram features (Mục 18)
So sánh TF-IDF unigram với TF-IDF unigram + bigram trên cùng IMDB split và cùng Logistic Regression.

Báo cáo vocabulary size, F1, accuracy, precision, recall và training time. `max_features=None` để thấy rõ feature space mở rộng khi thêm bigram; `min_df=2` bỏ n-gram chỉ xuất hiện một lần.

In [1]:
from pathlib import Path
import zipfile, re, html, time
import numpy as np
import pandas as pd

ZIP_PATH=Path('/content/imdb.zip')
if not ZIP_PATH.exists():
    from google.colab import files
    print('Chọn imdb.zip để tải lên Colab.')
    uploaded=files.upload()
    ZIP_PATH=Path(next(iter(uploaded)))

def load_imdb_zip(zip_path):
    frames=[]
    with zipfile.ZipFile(zip_path) as z:
        csvs=[n for n in z.namelist() if n.lower().endswith('.csv')]
        if not csvs:
            raise ValueError('Không tìm thấy CSV trong ZIP.')
        for name in csvs:
            with z.open(name) as f:
                part=pd.read_csv(f)
            part.columns=[str(c).strip().lower() for c in part.columns]
            if {'review','sentiment'}.issubset(part.columns):
                part=part[['review','sentiment']].copy()
            elif {'text','label'}.issubset(part.columns):
                part=part.rename(columns={'text':'review','label':'sentiment'})[['review','sentiment']].copy()
            else:
                raise ValueError(f'{name}: cần cột review/sentiment hoặc text/label.')
            frames.append(part)
    return pd.concat(frames,ignore_index=True)

df=load_imdb_zip(ZIP_PATH)
df['review']=df['review'].fillna('').astype(str)
df['sentiment']=df['sentiment'].astype(str).str.strip().str.lower()
df=df[df['sentiment'].isin(['positive','negative'])].drop_duplicates('review').reset_index(drop=True)
df['label']=df['sentiment'].map({'negative':0,'positive':1})

def normalize_review(text):
    text=html.unescape(text)
    text=re.sub(r'<[^>]+>',' ',text)
    return re.sub(r'\s+',' ',text).strip().lower()

df['text_clean']=df['review'].map(normalize_review)
print('Review sau loại trùng:',f'{len(df):,}')
print(df['sentiment'].value_counts().to_string())

Chọn imdb.zip để tải lên Colab.


Saving imdb.zip to imdb.zip
Review sau loại trùng: 49,582
sentiment
positive    24884
negative    24698


## Chia train/validation/test = 60/20/20
Dùng cùng `random_state=42` và stratify với các mục 14-17. Hai vectorizer được fit riêng trên train để tránh leakage; cùng validation/test được dùng để so sánh.

In [2]:
from sklearn.model_selection import train_test_split
X_trainval,X_test,y_trainval,y_test=train_test_split(
    df['text_clean'],df['label'],test_size=0.20,random_state=42,stratify=df['label']
)
X_train,X_val,y_train,y_val=train_test_split(
    X_trainval,y_trainval,test_size=0.25,random_state=42,stratify=y_trainval
)
print(f'Train={len(X_train):,}; Validation={len(X_val):,}; Test={len(X_test):,}')

Train=29,748; Validation=9,917; Test=9,917


## Huấn luyện hai mô hình
Model A dùng unigram `(1,1)`. Model B dùng unigram + bigram `(1,2)`; như vậy các feature `not good` và `very good` có thể được biểu diễn trực tiếp. Cả hai dùng cùng TF-IDF settings và Logistic Regression.

In [3]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score,precision_score,recall_score,f1_score,
                             confusion_matrix,classification_report)

def fit_and_evaluate(name,ngram_range):
    model=Pipeline([
        ('tfidf',TfidfVectorizer(ngram_range=ngram_range,min_df=2,
                                 strip_accents='unicode',sublinear_tf=True)),
        ('classifier',LogisticRegression(max_iter=1000,random_state=42))
    ])
    start=time.perf_counter()
    model.fit(X_train,y_train)
    train_seconds=time.perf_counter()-start
    y_pred=model.predict(X_test)
    vocab=model.named_steps['tfidf'].get_feature_names_out()
    result={
        'Model':name,
        'Features': 'Unigram' if ngram_range==(1,1) else 'Unigram + Bigram',
        'Vocabulary size':len(vocab),
        'Training time (s)':train_seconds,
        'Accuracy':accuracy_score(y_test,y_pred),
        'Precision':precision_score(y_test,y_pred,zero_division=0),
        'Recall':recall_score(y_test,y_pred,zero_division=0),
        'F1':f1_score(y_test,y_pred,zero_division=0)
    }
    print(f'\n{name} confusion matrix [[TN, FP], [FN, TP]]:')
    print(confusion_matrix(y_test,y_pred,labels=[0,1]))
    print(f'\n{name} classification report:')
    print(classification_report(y_test,y_pred,target_names=['negative','positive'],digits=4,zero_division=0))
    return model,result,set(vocab)

unigram_model,unigram_result,unigram_vocab=fit_and_evaluate('Model A - Unigram',(1,1))
uni_bigram_model,uni_bigram_result,uni_bigram_vocab=fit_and_evaluate('Model B - Uni + Bigram',(1,2))


Model A - Unigram confusion matrix [[TN, FP], [FN, TP]]:
[[4361  579]
 [ 445 4532]]

Model A - Unigram classification report:
              precision    recall  f1-score   support

    negative     0.9074    0.8828    0.8949      4940
    positive     0.8867    0.9106    0.8985      4977

    accuracy                         0.8967      9917
   macro avg     0.8971    0.8967    0.8967      9917
weighted avg     0.8970    0.8967    0.8967      9917


Model B - Uni + Bigram confusion matrix [[TN, FP], [FN, TP]]:
[[4363  577]
 [ 413 4564]]

Model B - Uni + Bigram classification report:
              precision    recall  f1-score   support

    negative     0.9135    0.8832    0.8981      4940
    positive     0.8878    0.9170    0.9022      4977

    accuracy                         0.9002      9917
   macro avg     0.9006    0.9001    0.9001      9917
weighted avg     0.9006    0.9002    0.9001      9917



## Bảng so sánh và kiểm tra bigram
Thời gian đo là fit TF-IDF + Logistic Regression trên train. Metrics được tính trên cùng test set; positive là lớp dương.

In [4]:
results=pd.DataFrame([unigram_result,uni_bigram_result])
display(results.style.format({
    'Training time (s)':'{:.2f}', 'Accuracy':'{:.4f}',
    'Precision':'{:.4f}', 'Recall':'{:.4f}', 'F1':'{:.4f}'
}))

added_features=uni_bigram_vocab-unigram_vocab
print(f'Feature mới khi thêm bigram: {len(added_features):,}')
for phrase in ['not good','very good']:
    print(f'\n"{phrase}" là feature của unigram+bigram?:',phrase in uni_bigram_vocab)
    print(f'"{phrase}" là feature của unigram?:',phrase in unigram_vocab)

f1_change=uni_bigram_result['F1']-unigram_result['F1']
print(f'\nF1 thay đổi: {f1_change:+.4f} ({f1_change*100:+.2f} điểm phần trăm)')

,Model,Features,Vocabulary size,Training time (s),Accuracy,Precision,Recall,F1
0,Model A - Unigram,Unigram,48484,9.41,0.8967,0.8867,0.9106,0.8985
1,Model B - Uni + Bigram,Unigram + Bigram,493564,28.84,0.9002,0.8878,0.9170,0.9022


Feature mới khi thêm bigram: 445,080

"not good" là feature của unigram+bigram?: True
"not good" là feature của unigram?: False

"very good" là feature của unigram+bigram?: True
"very good" là feature của unigram?: False

F1 thay đổi: +0.0037 (+0.37 điểm phần trăm)


## Trả lời câu hỏi mục 18
Bigram có thể cải thiện phân loại vì giữ được một phần ngữ cảnh cục bộ: `not good` khác với `very good`, trong khi unigram chỉ thấy riêng các từ `not`, `good`, `very`. Đổi lại, bigram làm vocabulary/feature space lớn hơn, tăng bộ nhớ và thời gian huấn luyện; nhiều bigram hiếm cũng có thể thêm nhiễu. Kết luận theo output thực tế: so sánh F1 cùng vocabulary size và training time; thêm bigram không đảm bảo F1 luôn tăng.